# 💥 Planetary Impact Visualisation

This notebook documents the pipeline for rendering particle mesh as examplifed by SPH planetary impact snapshots and visualising them as animated meshes in Blender.

---

## 📦 Pipeline Overview
```
SPH Snapshots (.hdf5) --> yt (dateset loading) --> AstroVis (data processing) --> Surface and Gas Data (.npz files) --> Blender
```


## 1. Imports & Configuration


In [ ]:
from AstroVis.backend import *
import yt
import numpy as np

res       = 1024                          # Voxel resolution of the density grid
input_dir = "star_impacts_snapshots_270"  

## 2. Data Processing

For each of the snapshots we run:

1. Load Snapshot by yt.load()
2. Convert the particle to grid by sph_to_grid()
3. Extract planet surface by grid_to_surface()
4. Filter emission gas by filter ()
5. Save Extracted data by save()

| Output | Method | Description |
|---|---|---|
| `planet_surface` | `grid_to_surface(threshold=0.003)` | Isosurface mesh at the density boundary separating solid/liquid planet from surrounding gas |
| `planet_emission_gas` | `planet.filter(densities > 0.003)` | Filter the low densities gas out|

The same threshold `0.003` is used for both so the surface mesh and emission gas particles align in the final render.

In [ ]:
planet_surface      = []
planet_emission_gas = []

for i in range(270):

    # --- Load snapshot ---
    ds     = yt.load(f"{input_dir}/demo_impact_n50_{i:04d}.hdf5")
    planet = load_particles(ds, ptype="PartType0")

    # --- SPH → uniform grid → isosurface mesh ---
    planet_grid = sph_to_grid(planet, field="density", res=res)
    planet_surface.append(
        grid_to_surface(planet_grid, threshold=0.003, field="density")
    )

    # --- Filter high-density emission gas particles ---
    emission_gas = planet.filter(planet.densities > 0.003)
    planet_emission_gas.append(emission_gas)

    print(f"Frame {i:03d} built successfully.")
    
data = {
    "planet_surface":      planet_surface,
    "planet_emission_gas": planet_emission_gas
}

save("planet_data.npz", data)

## 3. Setup Animation in Blender

> ⚠️ Run the cells below inside Blender's **Python Console** or **Text Editor** — `bpy` is only available there.

Reload `planet_data.npz` from disk and unpack the two frame arrays.  
`data['Particles']` holds the filtered SPH emission gas; `data['Surface']` holds the isosurface meshes.

Each dataset gets its own shader and animated mesh object in Blender.

| Object | Data | Shader | Visual Role |
|---|---|---|---|
| `gas` | `particle` | `gas` | Glowing emission plume — high-density ejecta particles |
| `surface` | `surface` | `surface` | Solid planet body — isosurface mesh boundary |

`create_mesh_shader()` builds a material node-tree for each role.  
`setup_mesh_animation()` creates a Blender mesh object and keyframes vertex positions across all 270 frames.  
`set_object_shader()` assigns the matching material to the object.


In [ ]:
# --- Load data for animation ---
from AstroVis.api import *

npz_folder = r"Your\Path\To\planet_data.npz"  # Update this path to where you saved the .npz file
data = load(npz_folder)

particle = data['Particles']['planet_emission_gas']
surface  = data['Surface']['planet_surface']

# --- Emission gas (particles) ---
create_mesh_shader("gas")
setup_mesh_animation(particle, "gas")
set_object_shader("gas")

# --- Planet surface (isosurface mesh) ---
create_mesh_shader("surface")
setup_mesh_animation(surface, "surface")
set_object_shader("surface")

print("Mesh animations and shaders set up. Press SPACE in the Blender viewport to preview the animation.")